# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/krishamerja1903/flyrank-ml-internship/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


### Finding 1 — Growing Content

The paper observed that growing content was approximately 37.6% longer and
20% younger than declining content.

**Methodology question:** Could factors such as topic type, competition,
existing search position, or content age explain part of the relationship
between word count and growth? Controlling for these factors could help
determine whether the observed relationship is associated with content length
itself or with differences between the groups.

### Finding 2 — Freshness Multiplier

The paper observed that the 31–90 day freshness window had a 7.88:1
growth-to-decline ratio.

**Methodology question:** Were pages selected for refresh independently of
their previous performance, or were stronger pages more likely to be updated?
If stronger pages were preferentially refreshed, selection bias could explain
part of the observed relationship. Comparing similar refreshed and
unrefreshed pages would provide stronger evidence.

In [1]:
paper_findings = {
    "Growing content": {
        "longer_pct": 37.6,
        "younger_pct": 20.0
    },
    "Freshness window": {
        "window": "31-90 days",
        "growth_to_decline_ratio": 7.88
    }
}

print("Paper findings selected for methodology audit:")
for finding, values in paper_findings.items():
    print(f"\n{finding}: {values}")

print("\nThese are reported paper findings; they are not causal claims.")

Paper findings selected for methodology audit:

Growing content: {'longer_pct': 37.6, 'younger_pct': 20.0}

Freshness window: {'window': '31-90 days', 'growth_to_decline_ratio': 7.88}

These are reported paper findings; they are not causal claims.


## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


### Before vs after validation

For the before comparison, I use a standard random 80/20 train-test split.
This can place pages from the same client in both training and test sets, so
client-specific patterns may leak across the evaluation boundary.

For the improved validation, I use an 80/20 split grouped by client. No client
appears in both training and test sets. This provides a more honest test of how
the model generalises to unseen clients.

Both approaches use the same Logistic Regression model, feature set, target,
and Precision@50 metric. Only the validation design changes.

In [6]:
# SECTION 2 — BEFORE/AFTER VALIDATION

from google.colab import userdata
from huggingface_hub import HfApi, hf_hub_download

import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split, GroupShuffleSplit
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression


# ---------------------------------
# 1. Load the same data used in W05
# ---------------------------------

HF_TOKEN = userdata.get("HF_TOKEN")

api = HfApi(token=HF_TOKEN)
print("Connected as:", api.whoami()["name"])

file_path = hf_hub_download(
    repo_id="FlyRank/internship-warehouse",
    filename="fact_content_daily_performance/month=2026-03/data_0.parquet",
    repo_type="dataset",
    token=HF_TOKEN
)

march = pd.read_parquet(file_path)

available = march[
    march["gsc_data_available"].eq(True) &
    march["ga4_data_available"].eq(True)
].copy()

available["report_date"] = pd.to_datetime(
    available["report_date"]
)

print("March rows:", len(march))
print("Available rows:", len(available))


# ---------------------------------
# 2. Recreate W05 model dataset
# ---------------------------------

early = available[
    available["report_date"].between(
        "2026-03-01", "2026-03-15"
    )
].copy()

late = available[
    available["report_date"].between(
        "2026-03-16", "2026-03-31"
    )
].copy()


early_page = (
    early.groupby(
        ["client_hash_id", "content_hash_id"],
        as_index=False
    )
    .agg(
        impressions=("gsc_impressions", "sum"),
        clicks=("gsc_clicks", "sum"),
        avg_position=("gsc_avg_position", "mean"),
        pageviews=("ga4_pageviews", "sum"),
        engaged_sessions=("ga4_engaged_sessions", "sum"),
        early_days=("report_date", "nunique")
    )
)


late_page = (
    late.groupby(
        ["client_hash_id", "content_hash_id"],
        as_index=False
    )
    .agg(
        late_clicks=("gsc_clicks", "sum"),
        late_days=("report_date", "nunique")
    )
)


model_df = early_page.merge(
    late_page,
    on=["client_hash_id", "content_hash_id"],
    how="inner"
)


model_df["early_avg_clicks"] = (
    model_df["clicks"] / model_df["early_days"]
)

model_df["late_avg_clicks"] = (
    model_df["late_clicks"] / model_df["late_days"]
)


# Future-window outcome — LABEL ONLY
model_df["declining"] = (
    model_df["late_avg_clicks"] <
    model_df["early_avg_clicks"]
).astype(int)


model_df["ctr"] = np.where(
    model_df["impressions"] > 0,
    model_df["clicks"] / model_df["impressions"],
    0
)


features = [
    "impressions",
    "clicks",
    "avg_position",
    "pageviews",
    "engaged_sessions"
]


model_df = model_df.replace(
    [np.inf, -np.inf], np.nan
)

model_df = model_df.dropna(
    subset=features
).copy()


print("Model rows:", len(model_df))
print(
    "Decline rate:",
    round(model_df["declining"].mean(), 4)
)


# ---------------------------------
# 3. Helper: train + Precision@50
# ---------------------------------

def fit_and_score(train_df, test_df):

    model = Pipeline([
        ("scaler", StandardScaler()),
        ("logreg", LogisticRegression(
            max_iter=1000,
            class_weight="balanced",
            random_state=42
        ))
    ])

    model.fit(
        train_df[features],
        train_df["declining"]
    )

    scored = test_df.copy()

    scored["model_score"] = model.predict_proba(
        scored[features]
    )[:, 1]

    top50 = scored.sort_values(
        "model_score",
        ascending=False
    ).head(50)

    precision50 = top50["declining"].mean()

    return model, scored, precision50


# ---------------------------------
# 4. BEFORE — ordinary random split
# ---------------------------------

random_train, random_test = train_test_split(
    model_df,
    test_size=0.20,
    random_state=42,
    stratify=model_df["declining"]
)

random_model, random_scored, random_p50 = fit_and_score(
    random_train,
    random_test
)

random_overlap = len(
    set(random_train["client_hash_id"]) &
    set(random_test["client_hash_id"])
)


# ---------------------------------
# 5. AFTER — grouped client split
# ---------------------------------

splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=42
)

train_idx, test_idx = next(
    splitter.split(
        model_df,
        model_df["declining"],
        groups=model_df["client_hash_id"]
    )
)

group_train = model_df.iloc[train_idx].copy()
group_test = model_df.iloc[test_idx].copy()


group_model, group_scored, group_p50 = fit_and_score(
    group_train,
    group_test
)

group_overlap = len(
    set(group_train["client_hash_id"]) &
    set(group_test["client_hash_id"])
)


# ---------------------------------
# 6. Before / After comparison
# ---------------------------------

comparison = pd.DataFrame({
    "Validation design": [
        "Random split (before)",
        "Grouped by client (after)"
    ],
    "Precision@50": [
        random_p50,
        group_p50
    ],
    "Client overlap": [
        random_overlap,
        group_overlap
    ],
    "Test rows": [
        len(random_test),
        len(group_test)
    ]
})


print("\nBEFORE / AFTER VALIDATION")
display(comparison)

print(
    "\nPrecision@50 change:",
    round(group_p50 - random_p50, 4)
)

print(
    "Grouped split client overlap:",
    group_overlap
)

Connected as: Krisha1903


fact_content_daily_performance/month=202(…): reconstructing file:   0%|          |  0.00B /  124MB            

fact_content_daily_performance/month=202(…): downloading bytes:           |  0.00B            

March rows: 9841378
Available rows: 364347
Model rows: 29394
Decline rate: 0.3643

BEFORE / AFTER VALIDATION


,Validation design,Precision@50,Client overlap,Test rows
0,Random split (before),0.74,27,5879
1,Grouped by client (after),0.70,0,1167



Precision@50 change: -0.04
Grouped split client overlap: 0


### Validation result

The random split achieved a Precision@50 of 0.74, but 27 clients appeared
in both the training and test sets.

After grouping the split by client, client overlap fell to zero and
Precision@50 was 0.70, a decrease of 0.04.

The small reduction suggests that the random split gave a slightly more
optimistic estimate. The grouped result is the more appropriate
decision-support estimate here because it evaluates the model on unseen
clients. This is an observed result for this dataset and split, not evidence
that the model will achieve the same performance in every setting.

## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


### Leakage audit

I audited the final feature set to check whether information from the future
outcome window, the target label, identifiers, or product-generated flags was
used as model input.

The model uses only impressions, clicks, average position, pageviews, and
engaged sessions calculated from the early March observation window.

The later-window click measurements are used only to construct the decline
label and are not model features. Client and content identifiers are used for
grouping and tracking, not as predictive features.

In [7]:
# SECTION 3 — LEAKAGE AUDIT

print("FINAL MODEL FEATURES:")
for feature in features:
    print("-", feature)

# Columns that must NOT be model inputs
forbidden_inputs = [
    "late_clicks",
    "late_days",
    "late_avg_clicks",
    "declining",
    "client_hash_id",
    "content_hash_id"
]

used_forbidden = [
    col for col in forbidden_inputs
    if col in features
]

print("\nLEAKAGE CHECK")
print("Forbidden inputs used:", used_forbidden)

if len(used_forbidden) == 0:
    print("Leakage audit: PASS")
else:
    print("Leakage audit: FAIL")

# Verify timing of feature and label windows
print("\nTIME-WINDOW CHECK")
print("Feature window: 2026-03-01 to 2026-03-15")
print("Outcome window: 2026-03-16 to 2026-03-31")

# Verify grouped validation
client_overlap = len(
    set(group_train["client_hash_id"]) &
    set(group_test["client_hash_id"])
)

print("\nGROUP CHECK")
print("Client overlap:", client_overlap)

if client_overlap == 0:
    print("Grouped validation check: PASS")
else:
    print("Grouped validation check: FAIL")

# Show real failure examples from the honest grouped split
group_scored["prediction"] = (
    group_scored["model_score"] >= 0.5
).astype(int)

false_positives = group_scored[
    (group_scored["prediction"] == 1) &
    (group_scored["declining"] == 0)
].copy()

false_negatives = group_scored[
    (group_scored["prediction"] == 0) &
    (group_scored["declining"] == 1)
].copy()

print("\nFAILURE EXAMPLES")
print("False positives:", len(false_positives))
print("False negatives:", len(false_negatives))

safe_cols = [
    "content_hash_id",
    "impressions",
    "clicks",
    "avg_position",
    "pageviews",
    "engaged_sessions",
    "model_score"
]

print("\nExample false positives:")
display(
    false_positives
    .sort_values("model_score", ascending=False)
    [safe_cols]
    .head(5)
)

print("\nExample false negatives:")
display(
    false_negatives
    .sort_values("model_score")
    [safe_cols]
    .head(5)
)

FINAL MODEL FEATURES:
- impressions
- clicks
- avg_position
- pageviews
- engaged_sessions

LEAKAGE CHECK
Forbidden inputs used: []
Leakage audit: PASS

TIME-WINDOW CHECK
Feature window: 2026-03-01 to 2026-03-15
Outcome window: 2026-03-16 to 2026-03-31

GROUP CHECK
Client overlap: 0
Grouped validation check: PASS

FAILURE EXAMPLES
False positives: 88
False negatives: 321

Example false positives:


,content_hash_id,impressions,clicks,avg_position,pageviews,engaged_sessions,model_score
23406,content_a47fd3545d1a68c6,2678,45,8.903121,102.0,2.0,0.914311
23360,content_30352f0b93c8b98d,1600,34,17.999995,62.0,4.0,0.854179
23388,content_7b50820e76b1a006,886,36,29.122832,68.0,1.0,0.831766
16414,content_f93cf81d5564765d,701,25,3.242482,26.0,0.0,0.763085
23441,content_ed1bd5af6c187f6f,1928,20,19.261200,54.0,5.0,0.745533



Example false negatives:


,content_hash_id,impressions,clicks,avg_position,pageviews,engaged_sessions,model_score
23348,content_0c29f92ee0a6987f,6,1,94.333333,2.0,0.0,0.279744
15040,content_4c7322376862be65,13,1,66.769231,1.0,0.0,0.332327
15259,content_c5c04445dec4a946,49,2,70.193333,2.0,0.0,0.337499
15937,content_f5c1fb735fb4a08c,7,1,59.000000,1.0,0.0,0.348015
23437,content_e4a6ed9b9592cb9c,6,1,56.000000,1.0,0.0,0.354170


## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


### Claim rewrite

**Original claim:**

The Logistic Regression model performs much better than the baseline and can
accurately identify pages that will decline.

**Rewritten evidence-safe claim:**

On the held-out client-grouped split, Logistic Regression achieved a measured
Precision@50 of 0.70, compared with 0.32 for the Week-4 rule-based baseline on
the same evaluation setup.

The validation audit also showed that Precision@50 decreased from 0.74 under
a random split to 0.70 under the stricter client-grouped split. The grouped
split had zero client overlap between training and test data.

The leakage audit found no future-window outcomes, target labels, identifiers,
or product flags in the final predictive feature set. However, the model still
produced 88 false positives and 321 false negatives at the 0.5 classification
threshold.

Therefore, the results provide directional evidence that the model may be
useful for ranking pages for review within this dataset. They should be
treated as decision-support evidence rather than proof that the model will
predict future decline equally well across other clients or time periods.

In [8]:
# SECTION 4 — CLAIM REWRITE CHECK

audit_summary = pd.DataFrame({
    "Check": [
        "Random split Precision@50",
        "Grouped split Precision@50",
        "Grouped client overlap",
        "Forbidden predictive inputs",
        "False positives",
        "False negatives"
    ],
    "Observed result": [
        round(random_p50, 2),
        round(group_p50, 2),
        group_overlap,
        len(used_forbidden),
        len(false_positives),
        len(false_negatives)
    ]
})

display(audit_summary)

print("\nFINAL CLAIM")
print(
    "Observed Precision@50 on the client-grouped split:",
    round(group_p50, 2)
)
print(
    "This result is directional and intended for decision support, "
    "not as a causal or universally generalisable claim."
)

,Check,Observed result
0,Random split Precision@50,0.74
1,Grouped split Precision@50,0.70
2,Grouped client overlap,0.00
3,Forbidden predictive inputs,0.00
4,False positives,88.00
5,False negatives,321.00



FINAL CLAIM
Observed Precision@50 on the client-grouped split: 0.7
This result is directional and intended for decision support, not as a causal or universally generalisable claim.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.